In [1]:
import numpy as np
from scipy.linalg import eigh
import matplotlib.pyplot as plt
from scipy.sparse.linalg import expm_multiply

In [ ]:
def heisenberg_hamiltonian_disorder(
    n,
    J=1.0,
    W=0.0,
    periodic=True,
    conserve_sz=False,
    nup=None,
    seed=None
):
    """
    Random-field Heisenberg model:
    H = J sum S_i · S_{i+1} + sum h_i S_i^z
    with h_i ~ Uniform[-W, W]
    """
    rng = np.random.default_rng(seed)
    h_fields = rng.uniform(-W, W, size=n)

    states, idx = all_basis_states(n, conserve_sz=conserve_sz, nup=nup)
    dim = len(states)
    H = np.zeros((dim, dim), dtype=np.float64)

    for k, s in enumerate(states):

        # Heisenberg exchange
        for bond in range(n if periodic else n - 1):
            j = (bond + 1) % n
            site1 = 1 << bond
            site2 = 1 << j
            di = site1 | site2
            da = s & di

            if da == di or da == 0:
                H[k, k] += 0.25 * J
            else:
                H[k, k] += -0.25 * J
                s2 = s ^ di
                if s2 in idx:
                    H[idx[s2], k] += 0.5 * J

        # Random field term (diagonal)
        for site in range(n):
            Sz = 0.5 if ((s >> site) & 1) else -0.5
            H[k, k] += h_fields[site] * Sz

    return H, states, idx, h_fields

def entanglement_entropy(psi, states, n, LA):
    """
    Von Neumann entanglement entropy of left LA sites
    """
    dimA = 1 << LA
    rhoA = np.zeros((dimA, dimA), dtype=complex)

    for i, si in enumerate(states):
        ai = si & ((1 << LA) - 1)
        bi = si >> LA
        for j, sj in enumerate(states):
            bj = sj >> LA
            if bi == bj:
                aj = sj & ((1 << LA) - 1)
                rhoA[ai, aj] += psi[i] * np.conj(psi[j])

    # Eigenvalues of reduced density matrix
    evals = np.linalg.eigvalsh(rhoA)
    evals = evals[evals > 1e-12]  # numerical cutoff

    return -np.sum(evals * np.log(evals))


def random_product_state(states, n, seed=None):
    rng = np.random.default_rng(seed)
    s = 0
    for i in range(n):
        if rng.random() < 0.5:
            s |= (1 << i)
    return s


if __name__ == '__main__':

    n = 8
    J = 1.0
    W = 3.0          # disorder strength (key MBL control parameter)
    periodic = True

    H, states, idx, h_fields = heisenberg_hamiltonian_disorder(
        n,
        J=J,
        W=W,
        periodic=periodic,
        conserve_sz=True,
        nup=n//2,
        seed=123
        )

    vals, vecs = diagonalize(H)

    print("Random fields h_i:")
    print(h_fields)

    print("Lowest 10 eigenvalues:")
    print(vals[:10])


In [ ]:
    # Parameters
if __name__ == '__main__':
        n = 8
        LA = n // 2
        J = 1.0
        W = 0.5        # try 0.5 (ergodic) and 5.0 (MBL)
        tmax = 20
        nt = 100

    # Hamiltonian
    H, states, idx, h_fields = heisenberg_hamiltonian_disorder(
        n,
        J=J,
        W=W,
        periodic=True,
        conserve_sz=True,
        nup=n//2,
        seed=42
    )

    vals, vecs = diagonalize(H)
    evolver = time_evolution_operator_from_eig(vals, vecs)

    # Initial random product state
    s0 = random_product_state(states, n, seed=1)
    idx0 = idx[s0]
    psi0 = basis_state_vector(idx0, len(states))

    times = np.linspace(0, tmax, nt)
    S_t = np.zeros(nt)

    for i, t in enumerate(times):
        psi_t = evolver(psi0, t)
        S_t[i] = entanglement_entropy(psi_t, states, n, LA)

    # Plot
    plt.figure(figsize=(7,4))
    plt.plot(times, S_t, label=f"W={W}")
    plt.xlabel("time")
    plt.ylabel("Entanglement entropy")
    plt.title("Entanglement growth after quench")
    plt.grid(True)
    plt.legend()
    plt.show()


## Parallelization of disorder effects

```python
for disorder realization r in parallel:
    generate h_i^(r)
    build H^(r)
    diagonalize H^(r)
    time evolve
    compute S(t)
average S(t) over r

seed = int(sys.argv[1])
S_t = run_single_realization((seed, ...))
np.save(f"S_seed{seed}.npy", S_t)

#SBATCH --array=0-99
#SBATCH --cpus-per-task=1

python run_realization.py $SLURM_ARRAY_TASK_ID


S_avg = np.mean([np.load(f) for f in files], axis=0)

```

In [ ]:
def run_single_realization(args):
    seed, n, J, W, tmax, nt = args
    # Build Hamiltonian
    H, states, idx, h_fields = heisenberg_hamiltonian_disorder(
        n,
        J=J,
        W=W,
        periodic=True,
        conserve_sz=True,
        nup=n//2,
        seed=seed
    )

    vals, vecs = diagonalize(H)
    evolver = time_evolution_operator_from_eig(vals, vecs)

    # Random initial product state
    s0 = random_product_state(states, n, seed=seed + 1000)
    psi0 = basis_state_vector(idx[s0], len(states))

    times = np.linspace(0, tmax, nt)
    LA = n // 2
    S_t = np.zeros(nt)

    for i, t in enumerate(times):
        psi_t = evolver(psi0, t)
        S_t[i] = entanglement_entropy(psi_t, states, n, LA)

    return S_t


if __name__ == '__main__':
    from multiprocessing import Pool, cpu_count
    n = 8
    J = 1.0
    W = 5.0
    tmax = 20
    nt = 100
    Nreal = 16   # number of disorder realizations
    args = [(r, n, J, W, tmax, nt) for r in range(Nreal)]
    with Pool(processes=cpu_count()) as pool:
        results = pool.map(run_single_realization, args)
    S_avg = np.mean(results, axis=0)
